In [23]:
import ee
import geemap #eemont
# ee.Authenticate()
# ee.Initialize()
from importlib import reload  

In [24]:
geemap.ee_initialize(project="water-sinapohlabeln")

Import modules

In [25]:

from modules import ms_indices_C02 as indices
from modules import configs, utils_string
from modules import nutils_Landsat_SR_C02_mask as utils_LS
from modules import nhigh_level_functions_C02_annual


Reloading

In [26]:
utils_string = reload(utils_string)
reload(indices)
reload(utils_LS)
reload(nhigh_level_functions_C02_annual)

<module 'modules.nhigh_level_functions_C02_annual' from '/isipd/projects/p_alex/ALEX-Development/2025-05_Praktikum-Sina/GEE_HotSpot/modules/nhigh_level_functions_C02_annual.py'>

In [ ]:
# PROPERTIES
# SET METADATA PARAMETERS
MAXCLOUD = 80 
STARTYEAR = 2005
ENDYEAR = 2024
STARTMONTH = 7
ENDMONTH = 8
SCALE = 30
longitudes = [-144, -145] #Besser: -155, #OG:-154 
latitudes = [65, 66] #Besser: 70.5#OG:70
SIZE_LON = 5
SIZE_LAT = 2

target_collection = 'projects/water-sinapohlabeln/assets/TCTrend_SR_2005-2024_TCVIS'

In [28]:
# image metadata Filters
config_trend = {
  'STARTYEAR': STARTYEAR,
  'ENDYEAR': ENDYEAR,
  'date_filter_yr' : ee.Filter.calendarRange(STARTYEAR, ENDYEAR, 'year'),
  'date_filter_mth' : ee.Filter.calendarRange(STARTMONTH, ENDMONTH, 'month'),
  'meta_filter_cld' : ee.Filter.lt('CLOUD_COVER', MAXCLOUD),
  'select_bands_visible' : ["SR_B1", "SR_B2","SR_B3","SR_B4"],
  'select_indices' : ["TCB", "TCG", "TCW"],
  'select_TCtrend_bands' : ["TCB_slope", "TCG_slope", "TCW_slope"],
  'geom' : None
}
#------ RUN FULL PROCESS FOR ALL REGIONS IN LOOP ------------------------------
#Map.addLayer(imageCollection, {}, 'TCVIS')

In [29]:
RUN = 1
m = geemap.Map()

In [30]:
for lowLat in latitudes:
    for leftLon in longitudes:
        
        
        # check for Hemisphere
        if lowLat < 0:
            sizeLat = SIZE_LAT * -1
        else:
            sizeLat = SIZE_LAT
            
        sizeLon = SIZE_LON
        
        # create Bounding Box
        geom = ee.Geometry.Polygon([leftLon,lowLat+sizeLat, leftLon, lowLat, leftLon+sizeLon, lowLat, leftLon+sizeLon, lowLat+sizeLat])
        config_trend['geom'] = geom
        m.addLayer(geom,{}, str(lowLat))

        #File name
        assetname_new = utils_string.make_TCTrendAssetNameSR(leftLon, lowLat, STARTYEAR, ENDYEAR)
        assetId = target_collection + '/' + assetname_new

        # Calculate Trend
        trend = nhigh_level_functions_C02_annual.nrunTCTrend(config_trend)
        if RUN:
            task = ee.batch.Export.image.toAsset(
                image=ee.Image(trend['visual']).toByte(),
                description=assetname_new,
                assetId=assetId,
                scale=SCALE,
                region=geom,
                maxPixels=1e12)

            task.start()



['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4']
['TCB', 'TCG', 'TCW']
['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4']
['TCB', 'TCG', 'TCW']
['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4']
['TCB', 'TCG', 'TCW']
['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4']
['TCB', 'TCG', 'TCW']


In [31]:
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…